In [1]:
import pandas as pd

In [2]:
beh=pd.read_csv(r'C:\Users\dimpu\Downloads\e-commerece project\datasets\Ecommerce.csv')
deliv=pd.read_csv(r'C:\Users\dimpu\Downloads\e-commerece project\datasets\Delivery_Logistics.csv')

In [3]:
# Ensuring matching indexes for safe structural binding

beh = beh.reset_index(drop=True)
deliv = deliv.reset_index(drop=True)

In [4]:
# Synchronise the keys and the timeline

deliv['session_id']=beh['session_id']
deliv['dispatch_date']=beh['visit_date']

In [5]:
beh.head()

,customer_id,session_id,visit_date,device_type,user_type,marketing_channel,product_id,product_category,unit_price,quantity,...,review_text,review_helpful_votes,payment_method,visit_day,visit_month,visit_weekday,visit_season,session_duration_bucket,revenue_normalized,location
0,1803,0,28-11-2024,2,1,2,894,6,651.57,1,...,1,0,1,28,11,3,0,Long,0.000000,209
1,7964,1,25-09-2024,2,0,4,844,2,945.27,4,...,1,0,2,25,9,2,0,Long,0.000000,213
2,6890,2,31-05-2024,1,1,0,865,0,400.44,4,...,1,0,2,31,5,4,1,Short,0.000000,10
3,4949,3,30-01-2024,1,0,2,851,3,1268.54,2,...,10,4,1,30,1,1,3,Very Long,0.305504,46
4,4896,4,25-02-2024,1,1,5,794,3,880.81,3,...,1,0,1,25,2,6,3,Very Short,0.000000,118


In [6]:
deliv.head()

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost,session_id,dispatch_date
0,250.99,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000008,no,delivered,3,1632.7206,0,28-11-2024
1,250.99,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003,no,delivered,5,640.1700,1,25-09-2024
2,250.99,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016,no,delivered,4,1448.1700,2,31-05-2024
3,250.99,dhl,electronics,ev van,same day,east,cold,269.7,12.69,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008,no,delivered,3,1486.5700,3,30-01-2024
4,250.99,dhl,clothing,van,two day,north,foggy,256.7,37.02,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000016,no,delivered,4,1394.5600,4,25-02-2024


# Mapping - converting num to name

In [7]:
# mapping

marketing_map = {
    0: "Direct", 1: "Email", 2: "Social Media", 3: "Referral/Affiliate",  
    4: "Paid Search", 5: "Organic Search" 
    }
beh['marketing_channel'] = beh['marketing_channel'].map(marketing_map)

# 2. Map Product Category
category_map = {
    0: "Electronics", 1: "Fashion", 2: "Home & Kitchen", 
    3: "Sports & Fitness", 4: "Books", 5: "Beauty & Care",
    6: "Footwear", 7: "Groceries" 
     
}
beh['product_category'] = beh['product_category'].map(category_map)

# 3. Map Payment Method
payment_map = {
    0: "UPI", 1: "Credit Card", 2: "Debit Card", 3: "Net Banking", 
    4: "Cash on Delivery", 5: "Wallet/BNPL"
}
beh['payment_method'] = beh['payment_method'].map(payment_map)

# Map device type
device_map={
    0 : 'Desktop', 1 : 'Mobile', 2: 'Tablet'
}
beh['device_type'] = beh['device_type'].map(device_map)

# Map user type
user_map={
    0 : 'New', 1 : 'Returning'
}
beh['user_type'] = beh['user_type'].map(user_map)

# map visit season
season ={
    0 : 'Autumn', 1 : 'Summer', 2 : 'Monsoon', 3 : 'Winter'
}

beh['visit_season']=beh['visit_season'].map(season)

# Normalization (dim and fact tables)

In [8]:
beh.head()

,customer_id,session_id,visit_date,device_type,user_type,marketing_channel,product_id,product_category,unit_price,quantity,...,review_text,review_helpful_votes,payment_method,visit_day,visit_month,visit_weekday,visit_season,session_duration_bucket,revenue_normalized,location
0,1803,0,28-11-2024,Tablet,Returning,Social Media,894,Footwear,651.57,1,...,1,0,Credit Card,28,11,3,Autumn,Long,0.000000,209
1,7964,1,25-09-2024,Tablet,New,Paid Search,844,Home & Kitchen,945.27,4,...,1,0,Debit Card,25,9,2,Autumn,Long,0.000000,213
2,6890,2,31-05-2024,Mobile,Returning,Direct,865,Electronics,400.44,4,...,1,0,Debit Card,31,5,4,Summer,Short,0.000000,10
3,4949,3,30-01-2024,Mobile,New,Social Media,851,Sports & Fitness,1268.54,2,...,10,4,Credit Card,30,1,1,Winter,Very Long,0.305504,46
4,4896,4,25-02-2024,Mobile,Returning,Organic Search,794,Sports & Fitness,880.81,3,...,1,0,Credit Card,25,2,6,Winter,Very Short,0.000000,118


In [9]:
# 5. Extract Dimensional Tables (Normalization)

dim_customers = beh.groupby('customer_id')[['location', 'user_type']] \
    .agg(lambda x: x.value_counts().idxmax()).reset_index()

dim_products = beh.groupby('product_id')[['product_category', 'unit_price']] \
    .agg(lambda x: x.value_counts().idxmax()).reset_index()

dim_logistics_partners = deliv[['delivery_partner', 'vehicle_type', 'package_type']].drop_duplicates()
dim_logistics_partners['partner_key'] = [f"PRTN-{i:03d}" for i in range(1, len(dim_logistics_partners) + 1)]

# Map partner key back to delivery fact table
deliv = deliv.merge(dim_logistics_partners, on=['delivery_partner', 'vehicle_type', 'package_type'])


In [10]:
# 6. Isolate Fact Tables
fact_sales = beh[['session_id', 'customer_id', 'product_id', 'visit_date', 'visit_season', 'quantity', 'discount_percent', 
                  'discount_amount', 'revenue', 'pages_viewed', 'time_on_site_sec', 'session_duration_bucket', 'added_to_cart', 
                  'rating', 'purchased', 'cart_abandoned', 'marketing_channel', 'device_type', 'payment_method']]

fact_logistics = deliv[['session_id', 'partner_key', 'dispatch_date', 'delivery_mode', 'region', 
                        'weather_condition', 'distance_km', 'package_weight_kg',
                         'delayed', 'delivery_status', 'delivery_cost', 'delivery_rating']]

# Decoding Location

In [11]:
# 1. Isolate the numerical location and the text region row-for-row
df = pd.DataFrame({
    'location_code': beh['location'],
    'region': deliv['region']
})

In [12]:
mapping_df = (
    df.groupby('location_code')['region']
    .agg(lambda x: x.value_counts().idxmax()) # Majority vote logic
    .reset_index()
)

mapping_df.shape

(225, 2)

In [13]:


# 2. Drop duplicates to create a clean, distinct lookup table
region_lookup = mapping_df.drop_duplicates().sort_values(by='location_code').reset_index(drop=True)

# 3. Create real Indian State and City labels for your dashboard slicers
# We map the 226 possible codes into structural geographic buckets
def generate_indian_states(row):
    code = row['location_code']
    reg = row['region']
    
    if reg == 'north':
        states = ['Delhi', 'Punjab', 'Uttar Pradesh', 'Haryana']

    elif reg == 'south':
        states = ['Telangana', 'Karnataka', 'Tamil Nadu', 'Andhra Pradesh']

    elif reg == 'west':
        states = ['Maharashtra', 'Gujarat', 'Goa']
 
    elif reg == 'east':
        states = ['West Bengal', 'Bihar', 'Odisha', 'Jharkhand']

    else: # Central
        states = ['Madhya Pradesh', 'Chhattisgarh']

        
    # Generate deterministic but varied labels based on the specific numeric code
    state = states[code % len(states)]
    
    return pd.Series([state])

# Apply the structural dictionary generation
region_lookup[['State']] = region_lookup.apply(generate_indian_states, axis=1)

print("Dim_Geography table successfully built and decoded!")


Dim_Geography table successfully built and decoded!


In [14]:
sales_df=beh[beh['purchased']==1]
delivery_df=fact_logistics[fact_logistics['session_id'].isin(sales_df['session_id'])]   

# Saving not done yet

In [15]:
'''
# 7. Generate clean export files for your BI Tool
dim_customers.to_csv("Dim_Customers.csv", index=False)
dim_products.to_csv("Dim_Products.csv", index=False)
dim_logistics_partners.to_csv("Dim_Logistics_Partners.csv", index=False)
fact_sales.to_csv("Fact_Sales.csv", index=False)

# Save this as Dim_Geography to include in your Star Schema model
region_lookup.to_csv("Dim_Geography.csv", index=False)


print("ETL Completed successfully! All data anomalies resolved.")

# deliveries for only purchased items
sales_df=beh[beh['purchased']==1]
delivery_df=fact_logistics[fact_logistics['session_id'].isin(sales_df['session_id'])]   
delivery_df.to_csv('Fact_delivery.csv', index=False)

'''


'\n# 7. Generate clean export files for your BI Tool\ndim_customers.to_csv("Dim_Customers.csv", index=False)\ndim_products.to_csv("Dim_Products.csv", index=False)\ndim_logistics_partners.to_csv("Dim_Logistics_Partners.csv", index=False)\nfact_sales.to_csv("Fact_Sales.csv", index=False)\n\n# Save this as Dim_Geography to include in your Star Schema model\nregion_lookup.to_csv("Dim_Geography.csv", index=False)\n\n\nprint("ETL Completed successfully! All data anomalies resolved.")\n\n# deliveries for only purchased items\nsales_df=beh[beh[\'purchased\']==1]\ndelivery_df=fact_logistics[fact_logistics[\'session_id\'].isin(sales_df[\'session_id\'])]   \ndelivery_df.to_csv(\'Fact_delivery.csv\', index=False)\n\n'

In [26]:
list = [dim_customers, dim_products, dim_logistics_partners, region_lookup, fact_sales, delivery_df]
for i,v in enumerate(list):
    print('='*25)
    print(i)
    print('shape :',v.shape)
    print('null values :',(v.isna().sum().sum()))
    print('duplicate values :',v.duplicated().sum())
    print('info :',v.info())

0
shape : (8442, 3)
null values : 0
duplicate values : 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8442 entries, 0 to 8441
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  8442 non-null   int64 
 1   location     8442 non-null   int64 
 2   user_type    8442 non-null   object
dtypes: int64(2), object(1)
memory usage: 198.0+ KB
info : None
1
shape : (899, 3)
null values : 0
duplicate values : 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 899 entries, 0 to 898
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        899 non-null    int64  
 1   product_category  899 non-null    object 
 2   unit_price        899 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 21.2+ KB
info : None
2
shape : (486, 4)
null values : 0
duplicate values : 0
<class 'pandas.core.frame.DataFrame'>
Inde